# Pretrained Attention-Only LLM + Full SSONN Adapter

This notebook is a simple experiment with a pretrained attention-only TransformerLens model.

Unlike the previous sweep, the SSONN block is not added as a residual delta. Each attention block is wrapped as:

```python
resid_post = self.base_block(resid, *args, **kwargs)
return self.delta(self.ln_delta(resid_post))
```

The pretrained attention block is not frozen. The SSONN adapter is inserted after attention. Training uses full language-modeling loss over every token in the context, while SSONN expansion still uses the library `EdgeFinder` / `expand_model` flow.

In [ ]:
# If needed, install the extra LLM dependencies first:
# %pip install transformer-lens datasets

In [ ]:
from pathlib import Path
import math
import os
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset
from datasets import load_dataset
from tqdm.auto import tqdm

from transformer_lens import HookedTransformer

import ssonn.train.train as ssonn_train
from ssonn.metrics.edge_finder import EdgeFinder
from ssonn.metrics.nonlinearity_metrics import AbsGradientEdgeMetric
from ssonn.model.model import ScaledNet

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
device

## Data

Use the same dataset family as `3b76cf790_ssonn_sweep_runner.py`: WikiText tokenized by the pretrained attention-only model tokenizer.

Each sample is now a full LM block:

```text
input:  tokens[t : t + seq_len]
target: tokens[t + 1 : t + seq_len + 1]
```

The notebook builds one token tensor per split, then `DataLoader` slices batches lazily. It does not materialize all batches as a Python list of tensors.

In [ ]:
MODEL_NAME = "attn-only-1l"
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-v1"
SEQ_LEN = 128
BATCH_SIZE = 4
NUM_WORKERS = 0
FINDER_BATCHES = 6

base = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
tokenizer = base.tokenizer
base.cfg.d_model, base.cfg.n_layers, base.cfg.d_vocab

In [ ]:
def encode_text(text):
    if not text:
        return []
    try:
        ids = tokenizer.encode(text, add_special_tokens=False)
    except TypeError:
        ids = tokenizer.encode(text)
    eos_id = getattr(tokenizer, "eos_token_id", None)
    if eos_id is not None:
        ids = list(ids) + [int(eos_id)]
    return list(ids)


def load_split_tokens(split):
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=split)
    token_buffer = []
    for row in tqdm(dataset, desc=f"Tokenizing {split}"):
        token_buffer.extend(encode_text(row.get("text", "")))
    return torch.tensor(token_buffer, dtype=torch.int32)


class TokenBlockDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len
        self.block = seq_len + 1

    def __len__(self):
        return int(self.tokens.numel() // self.block)

    def __getitem__(self, idx):
        start = idx * self.block
        block = self.tokens[start : start + self.block].long()
        return block[:-1].contiguous(), block[1:].contiguous()


train_tokens = load_split_tokens("train")
val_tokens = load_split_tokens("validation")

train_dataset = TokenBlockDataset(train_tokens, SEQ_LEN)
val_dataset = TokenBlockDataset(val_tokens, SEQ_LEN)
finder_dataset = Subset(
    train_dataset,
    range(min(FINDER_BATCHES * BATCH_SIZE, len(train_dataset))),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)
finder_loader = DataLoader(
    finder_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)

del base
if torch.cuda.is_available():
    torch.cuda.empty_cache()

len(train_dataset), len(val_dataset), len(train_loader), len(val_loader), len(finder_loader)

In [ ]:
print("train tokens:", int(train_tokens.numel()))
print("validation tokens:", int(val_tokens.numel()))
print("train LM blocks:", len(train_dataset))
print("validation LM blocks:", len(val_dataset))
print("train batches per epoch:", len(train_loader))
print("validation batches per epoch:", len(val_loader))
print("finder batches for expansion metric:", len(finder_loader))

x0, y0 = train_dataset[0]
print("sample x shape:", tuple(x0.shape))
print("sample y shape:", tuple(y0.shape))
print("first target equals shifted input:", bool(torch.equal(x0[1:], y0[:-1])))

## Model

The attention block remains inside the model and stays trainable. The SSONN adapter replaces the block output, not adds a residual delta.

In [ ]:
class TokenSSONNAdapter(nn.Module):
    def __init__(self, d_model, device):
        super().__init__()
        weight = torch.eye(d_model, device=device)
        bias = torch.zeros(d_model, device=device)
        self.scaled = ScaledNet(weight, bias, device=str(device))
        self.scaled.dropout.p = 0.0

    def forward(self, x):
        batch, seq, width = x.shape
        y = self.scaled(x.reshape(batch * seq, width))
        return y.reshape(batch, seq, width)


class AttentionBlockWithFullSSONNAdapter(nn.Module):
    def __init__(self, base_block, d_model, adapter):
        super().__init__()
        self.base_block = base_block
        self.ln_delta = nn.LayerNorm(d_model)
        self.delta = adapter

    def forward(self, resid, *args, **kwargs):
        resid_post = self.base_block(resid, *args, **kwargs)
        return self.delta(self.ln_delta(resid_post))


class AttentionOnlyLMWithSSONNAdapters(nn.Module):
    def __init__(self, model_name, device):
        super().__init__()
        self.backbone = HookedTransformer.from_pretrained(model_name, device=device)
        d_model = int(self.backbone.cfg.d_model)

        self.ssonn_layer_names = []
        for idx, block in enumerate(self.backbone.blocks):
            adapter = TokenSSONNAdapter(d_model, device)
            layer_name = f"ssonn_adapter_{idx}"
            setattr(self, layer_name, adapter.scaled)
            self.ssonn_layer_names.append(layer_name)
            self.backbone.blocks[idx] = AttentionBlockWithFullSSONNAdapter(block, d_model, adapter).to(device)

    def forward(self, tokens):
        return self.backbone(tokens)


model = AttentionOnlyLMWithSSONNAdapters(MODEL_NAME, device).to(device)

# Attention is not frozen: every parameter remains trainable.
for parameter in model.parameters():
    parameter.requires_grad = True

model.ssonn_layer_names

## Initial Weight Mode

Use `INITIAL_WEIGHT_MODE = "full_reinit"` if the LLM should be trained from random transformer-style initialization instead of pretrained weights. This is different from CNN-style Kaiming init: TransformerLens attention parameters are mostly raw `nn.Parameter`s, not `nn.Linear` modules, so `model.apply(init_weights)` would miss important matrices and use the wrong distribution.

In [ ]:
INITIAL_WEIGHT_MODE = "full_reinit"  # "pretrained" or "full_reinit"
TRANSFORMER_INIT_STD = 0.02


def reset_ssonn_adapters_to_identity(model):
    for name in model.ssonn_layer_names:
        layer = getattr(model, name)
        with torch.no_grad():
            layer.weight.zero_()
            diagonal = min(layer.out_features, layer.in_features)
            layer.weight[:diagonal, :diagonal] = torch.eye(
                diagonal, device=layer.weight.device, dtype=layer.weight.dtype
            )
            layer.bias.zero_()


def reinitialize_transformer_backbone(model, std=0.02):
    with torch.no_grad():
        for name, param in model.backbone.named_parameters():
            if not param.requires_grad:
                continue
            if param.ndim == 1 or name.split(".")[-1].startswith("b_") or name.endswith(".b"):
                param.zero_()
            else:
                nn.init.normal_(param, mean=0.0, std=std)

    for module in model.modules():
        if isinstance(module, nn.LayerNorm):
            with torch.no_grad():
                if module.elementwise_affine:
                    module.weight.fill_(1.0)
                    module.bias.zero_()


if INITIAL_WEIGHT_MODE == "pretrained":
    print("Using pretrained HookedTransformer weights.")
elif INITIAL_WEIGHT_MODE == "full_reinit":
    reinitialize_transformer_backbone(model, std=TRANSFORMER_INIT_STD)
    reset_ssonn_adapters_to_identity(model)
    print(f"Reinitialized HookedTransformer backbone with std={TRANSFORMER_INIT_STD}.")
else:
    raise ValueError(f"Unknown INITIAL_WEIGHT_MODE: {INITIAL_WEIGHT_MODE}")

## SSONN Training Setup

This follows the same high-level lifecycle as `experiments/training/cifar10.ipynb`: train, evaluate, periodically run SSONN expansion with `EdgeFinder`, reinitialize according to `REINIT_MODE`, then rebuild the optimizer.

The loop is local to this notebook because the library `train_ssonn` expects ordinary classification tensors shaped `[batch, classes]`, while LM training needs logits shaped `[batch, seq, vocab]` and targets shaped `[batch, seq]`.

There are two useful reinitialization modes after expansion:

- `REINIT_MODE = "ssonn_only"`: reinitialize only `ScaledNet`, leaving the current attention/backbone weights untouched.
- `REINIT_MODE = "full"`: reinitialize the whole HookedTransformer backbone with the same transformer-style initialization used by `INITIAL_WEIGHT_MODE = "full_reinit"`, then reset SSONN adapters to identity.

Run the notebook twice with different `REINIT_MODE` values to compare both hypotheses.

In [ ]:
REINIT_MODE = "ssonn_only"  # "ssonn_only" or "full"


def ssonn_only_weights_init(module):
    if isinstance(module, ScaledNet):
        module.reinitialize_parameters()


def transformer_full_weights_init(module):
    if isinstance(module, AttentionOnlyLMWithSSONNAdapters):
        reinitialize_transformer_backbone(module, std=TRANSFORMER_INIT_STD)
        reset_ssonn_adapters_to_identity(module)


if REINIT_MODE == "ssonn_only":
    ssonn_train.weights_init = ssonn_only_weights_init
elif REINIT_MODE == "full":
    ssonn_train.weights_init = transformer_full_weights_init
else:
    raise ValueError(f"Unknown REINIT_MODE: {REINIT_MODE}")

print("Reinitialization mode:", REINIT_MODE)


def lm_cross_entropy(logits, targets):
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
    )


criterion = lm_cross_entropy

hyperparams = {
    "num_epochs": 8,
    "batch_size": BATCH_SIZE,
    "edge_importance_metric": AbsGradientEdgeMetric(criterion),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {name: 0.85 for name in model.ssonn_layer_names},
    "pruning_thresholds": {name: 1.0 for name in model.ssonn_layer_names},
    "learning_rate": 2e-5,
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 64,
    "weight_decay": 0.0,
    "t_max": 2,
}


def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams["learning_rate"]
    if current_lr is None:
        current_lr = base_lr

    optimizer = optim.AdamW(
        model_instance.parameters(),
        lr=current_lr,
        weight_decay=hyperparams["weight_decay"],
    )
    for group in optimizer.param_groups:
        group["initial_lr"] = base_lr
    return optimizer


hyperparams

## Comet Logging

Set `COMET_API_KEY`, `COMET_WORKSPACE`, and optionally `COMET_PROJECT_NAME` in the environment to enable live logging.

In [ ]:
experiment = None

if os.environ.get("COMET_API_KEY") and os.environ.get("COMET_WORKSPACE"):
    from comet_ml import start

    experiment = start(
        api_key=os.environ.get("COMET_API_KEY"),
        workspace=os.environ.get("COMET_WORKSPACE"),
        project_name=os.environ.get("COMET_PROJECT_NAME", "ssonn-experiments"),
    )
    experiment.set_name(
        f"attention_full_ssonn_{MODEL_NAME}_reinit_{REINIT_MODE}_seq{SEQ_LEN}_seed0"
    )
    experiment.log_parameters(
        {
            **hyperparams,
            "model_name": MODEL_NAME,
            "dataset_name": DATASET_NAME,
            "dataset_config": DATASET_CONFIG,
            "seq_len": SEQ_LEN,
            "batch_size": BATCH_SIZE,
            "train_tokens": int(train_tokens.numel()),
            "validation_tokens": int(val_tokens.numel()),
            "train_lm_blocks": len(train_dataset),
            "validation_lm_blocks": len(val_dataset),
            "train_batches_per_epoch": len(train_loader),
            "validation_batches_per_epoch": len(val_loader),
            "finder_batches": len(finder_loader),
            "finder_split": "train_subset",
            "full_lm_loss": True,
            "reinit_mode": REINIT_MODE,
            "initial_weight_mode": INITIAL_WEIGHT_MODE,
            "transformer_init_std": TRANSFORMER_INIT_STD,
            "ssonn_layer_names": model.ssonn_layer_names,
        }
    )
else:
    print("Comet logging disabled: set COMET_API_KEY and COMET_WORKSPACE to enable it.")

experiment

### Metric sanity note

Training now uses the full WikiText train split through a lazy `DataLoader`, and validation uses the full validation split. Token accuracy is measured over all predicted positions in each `[batch, seq]` target, not only over the last token.

In [ ]:
def token_accuracy(logits, targets):
    preds = logits.argmax(dim=-1)
    return (preds == targets).sum().item(), targets.numel()


def train_lm_one_epoch(model, train_loader, optimizer, criterion, device, max_norm=1.0):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc="Train", leave=False)
    for inputs, targets in pbar:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        optimizer.step()

        batch_correct, batch_total = token_accuracy(logits.detach(), targets)
        total_loss += loss.item()
        correct += batch_correct
        total += batch_total
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct / total:.4f}"})

    return total_loss / len(train_loader), correct / total


def eval_lm_one_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc="Val", leave=False):
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)

            batch_correct, batch_total = token_accuracy(logits, targets)
            total_loss += loss.item()
            correct += batch_correct
            total += batch_total

    return total_loss / len(val_loader), correct / total


def count_model_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    nonzero = sum((p.detach() != 0).sum().item() for p in model.parameters())
    return total, trainable, nonzero


def train_ssonn_lm(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment=None,
):
    model = model.to(device)
    optimizer = optimizer_factory(model)
    ssonn_train.apply_sparsity_mask(model)

    edge_finder = EdgeFinder(
        metric=hyperparams["edge_importance_metric"],
        dataloader=finder_loader,
        device=device,
        aggregation_mode=hyperparams.get("edge_score_aggregation", "mean"),
        max_to_replace=hyperparams.get("max_new_edges_per_expansion"),
    )
    sched = CosineAnnealingLR(optimizer, T_max=hyperparams["t_max"], eta_min=0, last_epoch=-1)

    print("--- Starting SSONN LM Training ---")
    for epoch in range(hyperparams["num_epochs"]):
        t_loss, t_acc = train_lm_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_acc = eval_lm_one_epoch(model, val_loader, criterion, device)

        if ((epoch - hyperparams["t_max"]) % (2 * hyperparams["t_max"])) == 0 and epoch != 0:
            print("   -> Action: Expanding Model.")
            expanded, num_new = ssonn_train.expand_model(
                model,
                edge_finder,
                hyperparams["expansion_thresholds"],
            )
            if expanded:
                print("   -> Re-initializing according to REINIT_MODE.")
                model.apply(ssonn_train.weights_init)

                current_lr = optimizer.param_groups[0]["lr"]
                optimizer = optimizer_factory(
                    model,
                    current_lr=current_lr,
                    base_lr=hyperparams["learning_rate"],
                )
                sched = CosineAnnealingLR(
                    optimizer,
                    T_max=hyperparams["t_max"],
                    eta_min=0,
                    last_epoch=epoch,
                )
                ssonn_train.apply_sparsity_mask(model)

                if experiment is not None:
                    experiment.log_metric("expansion_event", num_new, step=epoch)
            else:
                print("   -> Expansion attempted but no edges selected.")

        sched.step()

        total_params, trainable_params, nonzero_params = count_model_params(model)
        lrs = [group["lr"] for group in optimizer.param_groups]
        lr_str = " | ".join(f"{lr:.1e}" for lr in lrs)
        ppl = math.exp(min(v_loss, 50.0))

        print(
            f"Epoch {epoch + 1}/{hyperparams['num_epochs']} | "
            f"T-Loss: {t_loss:.4f} | V-Loss: {v_loss:.4f} | "
            f"T-Acc: {t_acc:.4f} | V-Acc: {v_acc:.4f} | "
            f"V-PPL: {ppl:.2f} | LRs: [{lr_str}] | "
            f"Params: {nonzero_params}/{total_params} trainable={trainable_params}"
        )

        if experiment is not None:
            experiment.log_metrics(
                {
                    "train_loss": t_loss,
                    "val_loss": v_loss,
                    "train_token_accuracy": t_acc,
                    "val_token_accuracy": v_acc,
                    "val_perplexity": ppl,
                    "lr": optimizer.param_groups[0]["lr"],
                    "total_params": total_params,
                    "trainable_params": trainable_params,
                    "nonzero_params": nonzero_params,
                    "sparsity": 1.0 - (nonzero_params / total_params),
                },
                step=epoch,
            )

    print("--- Training Finished ---")
    return model


trained_model = train_ssonn_lm(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment=experiment,
)

In [ ]:
for name in trained_model.ssonn_layer_names:
    layer = getattr(trained_model, name)
    print(name, "in_features=", layer.in_features, "sub_layers=", len(layer.sub_linears))